In [187]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import re #regex
import nltk #natural language tool kit
import spacy #another
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer #size

import tensorflow as tf
from tensorflow.keras.utils import to_categorical

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report,accuracy_score

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout,SpatialDropout1D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

In [30]:
pd.set_option('display.max_columns',None)

In [31]:
df=pd.read_csv('/content/Sentiment.csv')
df.head(2)

,id,candidate,candidate_confidence,relevant_yn,relevant_yn_confidence,sentiment,sentiment_confidence,subject_matter,subject_matter_confidence,candidate_gold,name,relevant_yn_gold,retweet_count,sentiment_gold,subject_matter_gold,text,tweet_coord,tweet_created,tweet_id,tweet_location,user_timezone
0,1,No candidate mentioned,1.0,yes,1.0,Neutral,0.6578,None of the above,1.0,NaN,I_Am_Kenzi,NaN,5,NaN,NaN,RT @NancyLeeGrahn: How did everyone feel about the Climate Change question last night? Exactly. #GOPDebate,NaN,2015-08-07 09:54:46 -0700,629697200650592256,NaN,Quito
1,2,Scott Walker,1.0,yes,1.0,Positive,0.6333,None of the above,1.0,NaN,PeacefulQuest,NaN,26,NaN,NaN,RT @ScottWalker: Didn't catch the full #GOPdebate last night. Here are some of Scott's best lines in 90 seconds. #Walker16 http://t.co/ZSfF…,NaN,2015-08-07 09:54:46 -0700,629697199560069120,NaN,NaN


In [32]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13871 entries, 0 to 13870
Data columns (total 21 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   id                         13871 non-null  int64  
 1   candidate                  13775 non-null  object 
 2   candidate_confidence       13871 non-null  float64
 3   relevant_yn                13871 non-null  object 
 4   relevant_yn_confidence     13871 non-null  float64
 5   sentiment                  13871 non-null  object 
 6   sentiment_confidence       13871 non-null  float64
 7   subject_matter             13545 non-null  object 
 8   subject_matter_confidence  13871 non-null  float64
 9   candidate_gold             28 non-null     object 
 10  name                       13871 non-null  object 
 11  relevant_yn_gold           32 non-null     object 
 12  retweet_count              13871 non-null  int64  
 13  sentiment_gold             15 non-null     obj

In [33]:
df.shape

(13871, 21)

In [34]:
df_filterd = df[['candidate','sentiment','text']]
df_filterd.head()

,candidate,sentiment,text
0,No candidate mentioned,Neutral,RT @NancyLeeGrahn: How did everyone feel about the Climate Change question last night? Exactly. #GOPDebate
1,Scott Walker,Positive,RT @ScottWalker: Didn't catch the full #GOPdebate last night. Here are some of Scott's best lines in 90 seconds. #Walker16 http://t.co/ZSfF…
2,No candidate mentioned,Neutral,RT @TJMShow: No mention of Tamir Rice and the #GOPDebate was held in Cleveland? Wow.
3,No candidate mentioned,Positive,RT @RobGeorge: That Carly Fiorina is trending -- hours after HER debate -- above any of the men in just-completed #GOPdebate says she's on …
4,Donald Trump,Positive,RT @DanScavino: #GOPDebate w/ @realDonaldTrump delivered the highest ratings in the history of presidential debates. #Trump2016 http://t.co…


In [35]:
df_filterd.shape

(13871, 3)

In [36]:
df_filterd.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13871 entries, 0 to 13870
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   candidate  13775 non-null  object
 1   sentiment  13871 non-null  object
 2   text       13871 non-null  object
dtypes: object(3)
memory usage: 325.2+ KB


In [37]:
df_filterd.isnull().sum()

,0
candidate,96
sentiment,0
text,0


In [38]:
df_filterd.dropna(inplace=True)

/tmp/ipykernel_3528/510533945.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filterd.dropna(inplace=True)


In [39]:
df_filterd.isnull().sum()

,0
candidate,0
sentiment,0
text,0


In [40]:
df_filterd.shape

(13775, 3)

In [41]:
df_filterd['text'][0]

'RT @NancyLeeGrahn: How did everyone feel about the Climate Change question last night? Exactly. #GOPDebate'

In [42]:
df_filterd['text'][1]

"RT @ScottWalker: Didn't catch the full #GOPdebate last night. Here are some of Scott's best lines in 90 seconds. #Walker16 http://t.co/ZSfF…"

In [46]:
df_filterd['text'][1000]

'Executive Experience Matters\n\n#gopdebate #ccot #gop #teaparty #imwithhuck #tcot #th2016 #RenewUS #wakeupamerica https://t.co/uCA5oguT29'

In [47]:
df_filterd['text'][32]

'The First #GOPDebate: Social Media Reaction and More http://t.co/X6KUVSkltF'

In [48]:
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [59]:
stop_words = set(stopwords.words("english"))

In [53]:
df_filterd['text'].iloc[78]

'RT @GovMikeHuckabee: Luntz: "As the lines go up, they almost reach 100. It almost never happens in the dial research we do." #GOPDebate\nhtt…'

In [60]:
def clean_tweet(text):
  text= str(text).lower()
  text = re.sub(r"https?://\S+|www\.\S+",'',text)
  text= re.sub(r'@\w+','',text)
  text = re.sub(r'#(\w+)',r'\1',text)
  text= re.sub(r'<[^>]+>',' ',text)
  text= re.sub(r'[^a-z\s]','',text)

  words = text.split()
  filtered_words = [word for word in words if word not in stop_words]
  text= " ".join(filtered_words)

  return text

In [61]:
df_filterd['clean_text']=df_filterd['text'].apply(clean_tweet)

/tmp/ipykernel_3528/1326592183.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filterd['clean_text']=df_filterd['text'].apply(clean_tweet)


In [62]:
df_filterd.head()

,candidate,sentiment,text,clean_text
0,No candidate mentioned,Neutral,RT @NancyLeeGrahn: How did everyone feel about the Climate Change question last night? Exactly. #GOPDebate,rt everyone feel climate change question last night exactly gopdebate
1,Scott Walker,Positive,RT @ScottWalker: Didn't catch the full #GOPdebate last night. Here are some of Scott's best lines in 90 seconds. #Walker16 http://t.co/ZSfF…,rt didnt catch full gopdebate last night scotts best lines seconds walker
2,No candidate mentioned,Neutral,RT @TJMShow: No mention of Tamir Rice and the #GOPDebate was held in Cleveland? Wow.,rt mention tamir rice gopdebate held cleveland wow
3,No candidate mentioned,Positive,RT @RobGeorge: That Carly Fiorina is trending -- hours after HER debate -- above any of the men in just-completed #GOPdebate says she's on …,rt carly fiorina trending hours debate men justcompleted gopdebate says shes
4,Donald Trump,Positive,RT @DanScavino: #GOPDebate w/ @realDonaldTrump delivered the highest ratings in the history of presidential debates. #Trump2016 http://t.co…,rt gopdebate w delivered highest ratings history presidential debates trump


In [63]:
df_filterd['clean_text'].iloc[7]

'going msnbc live around pm et gopdebate'

In [64]:
df_filterd['text'].iloc[7]

'Going on #MSNBC Live with @ThomasARoberts around 2 PM ET.  #GOPDebate'

In [65]:
df_filterd['sentiment'].value_counts() #class imbalance, downsampling to 2222

,count
sentiment,
Negative,8447
Neutral,3106
Positive,2222


In [66]:
#Sentiment distribution
sentiment_counts = df_filterd['sentiment'].value_counts().sort_index()
for sentiment, count in sentiment_counts.items():
  perc = (count/len(df_filterd))*100
  print(f"{sentiment} : {perc:.2f}%")

#biased more data -> negative

Negative : 61.32%
Neutral : 22.55%
Positive : 16.13%


In [70]:
#handle class imbalance using downsampling
from sklearn.utils import resample

df_neg=df_filterd[df_filterd['sentiment']=='Negative']
df_neu = df_filterd[df_filterd['sentiment']=='Neutral']
df_pos = df_filterd[df_filterd['sentiment']=='Positive']

target_size=len(df_pos)

df_neg_downsampled = resample(df_neg,replace=False,n_samples=target_size, random_state=42)
df_neu_downsampled = resample(df_neu,replace=False,n_samples=target_size, random_state=42)

df_balanced = pd.concat([df_neg_downsampled,df_neu_downsampled,df_pos])

df_balanced = df_balanced.sample(frac=1,random_state=42).reset_index(drop=True)

In [163]:
df_balanced['sentiment'].value_counts()

,count
sentiment,
Negative,2222
Neutral,2222
Positive,2222


In [173]:
label_mapping ={'Negative':0,'Neutral':1,'Positive':2}
y_integers = df_balanced['sentiment'].map(label_mapping).values

In [176]:
y_integers.shape

(6666,)

In [177]:
y_integers

array([0, 1, 2, ..., 2, 2, 0])

In [164]:
MAX_FEATURES = 10000 #vocabulary only top 10000 most frequent words in dataset
MAX_LEN = 100 #length of each tweet. pad or truncate

In [165]:
X = df_balanced['clean_text'].values #features
y = df_balanced['sentiment'].values #target

In [166]:
tokenizer = Tokenizer(num_words=MAX_FEATURES,oov_token='<OOV>',split=' ')
tokenizer.fit_on_texts(X)

In [167]:
#convert sent in to sequences
X_sequences = tokenizer.texts_to_sequences(X)

In [168]:
X_sequences[0]

[3, 141, 48, 13, 54, 184, 46, 257, 299, 83, 269, 108, 9, 138, 122]

In [169]:
X_sequences[3]

[167, 1394, 825, 1064, 3761, 1642, 3762, 450, 581, 663, 3763, 3764, 2, 984]

In [188]:
X_padded = pad_sequences(X_sequences, maxlen=MAX_LEN, padding='pre',truncating='pre')

In [189]:
X_train,X_test, y_train_text, y_test_text = train_test_split(
    X_padded, y_integers,
    test_size=0.20,
    stratify = y_integers,
    random_state=42
)

In [190]:
y_train_text[3]

np.int64(1)

In [191]:
y_train = to_categorical(y_train_text,num_classes=3)
y_test = to_categorical(y_test_text,num_classes=3)

In [192]:
X_train.shape

(5332, 100)

In [193]:
y_train.shape

(5332, 3)

In [194]:
y_test.shape

(1334, 3)

In [215]:
def create_LSTM_model():
  model= Sequential([
      Embedding(input_dim=MAX_FEATURES, output_dim=128, input_length=MAX_LEN),
      SpatialDropout1D(0.4),
      LSTM(units=64,return_sequences=False),
      Dropout(0.4),
      #Dense(32,activation='relu'),
      Dense(3,activation='softmax')
  ])
  return model

In [216]:
lstm_model = create_LSTM_model()

#compile the lstm nn
lstm_model.compile(
    optimizer = 'adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [217]:
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience = 2,
    restore_best_weights=True
)

In [218]:
lstm_history = lstm_model.fit(
    X_train, y_train,
    batch_size=128,
    epochs = 10,
    validation_split=0.20,
    callbacks=[early_stopping]
)

Epoch 1/10
34/34 ━━━━━━━━━━━━━━━━━━━━ 13s 301ms/step - accuracy: 0.4251 - loss: 1.0809 - val_accuracy: 0.5286 - val_loss: 1.0391
Epoch 2/10
34/34 ━━━━━━━━━━━━━━━━━━━━ 10s 284ms/step - accuracy: 0.5461 - loss: 0.9746 - val_accuracy: 0.5764 - val_loss: 0.9228
Epoch 3/10
34/34 ━━━━━━━━━━━━━━━━━━━━ 8s 226ms/step - accuracy: 0.6814 - loss: 0.7862 - val_accuracy: 0.6214 - val_loss: 0.8708
Epoch 4/10
34/34 ━━━━━━━━━━━━━━━━━━━━ 10s 285ms/step - accuracy: 0.7733 - loss: 0.6099 - val_accuracy: 0.6092 - val_loss: 0.8894
Epoch 5/10
34/34 ━━━━━━━━━━━━━━━━━━━━ 10s 283ms/step - accuracy: 0.8251 - loss: 0.4773 - val_accuracy: 0.6092 - val_loss: 0.9195


In [219]:
test_loss,test_accuracy = lstm_model.evaluate(X_test,y_test)
print(f'Loss: {test_loss:.2f}')
print(f'Accuracy: {test_accuracy:.2f}')

42/42 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - accuracy: 0.5652 - loss: 0.9358
Loss: 0.94
Accuracy: 0.57


In [221]:
# 1. Get the model's raw probability guesses for the test data
y_pred_probabilities = lstm_model.predict(X_test)

# 2. Convert probabilities and true targets back into 1D integers (0, 1, or 2)
y_pred_labels = np.argmax(y_pred_probabilities, axis=1)
y_true_labels = np.argmax(y_test, axis=1)

# 3. Print the text evaluation report
class_names = ['Negative', 'Neutral', 'Positive']
print("==================================================")
print("             CLASSIFICATION REPORT               ")
print("==================================================")
print(classification_report(y_true_labels, y_pred_labels, target_names=class_names))

print("\n==================================================")
print("               CONFUSION MATRIX                  ")
print("==================================================")
print(confusion_matrix(y_true_labels, y_pred_labels))

42/42 ━━━━━━━━━━━━━━━━━━━━ 3s 56ms/step
             CLASSIFICATION REPORT               
              precision    recall  f1-score   support

    Negative       0.60      0.58      0.59       444
     Neutral       0.48      0.57      0.52       445
    Positive       0.64      0.55      0.59       445

    accuracy                           0.57      1334
   macro avg       0.57      0.57      0.57      1334
weighted avg       0.57      0.57      0.57      1334


               CONFUSION MATRIX                  
[[256 129  59]
 [112 252  81]
 [ 58 141 246]]
